# Clean (overlap-filtered) OS-data perplexity -- all languages, all models

BabyLM's OpenSubtitles training data overlaps with the "OS-data" evaluation set (OPUS-100 en-hi/en-te) -- confirmed ~25% for the English side (against `open_subtitles.train.txt`), and this notebook additionally checks the Hindi side (against `translated-babylm-hindi/train/open_subtitles.train.hi.txt`) and Telugu side (against `translated-babylm-telugu/train/open_subtitles.train.te.txt`).

For each language, filters out any OPUS-100 line that exactly matches a BabyLM training line, then recomputes perplexity on the remaining clean subset -- for every monolingual and bilingual model that has an OS-data number in `evaluation.md`.

In [ ]:
# Cell 1: clone the repo and install dependencies
import os

if not os.path.isdir("/content/BabyLM"):
    !git clone https://github.com/vishnup22/BabyLM.git /content/BabyLM

%cd /content/BabyLM
!git checkout evaluation
!git pull
!pip install -q torch transformers accelerate datasets huggingface_hub requests

In [ ]:
# Cell 2: log in to Hugging Face (needed for gated meta-llama/Llama-3.2-1B and private repos)
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!hf auth whoami

In [ ]:
# Cell 3: build the overlap set for each language, from that language's own BabyLM
# OpenSubtitles training file (English: original BabyLM training data; Hindi/Telugu: the
# GPT-5-mini translated equivalents used to train the Hindi/Telugu models)
import re
import requests

BABYLM_OS_URLS = {
    "en": "https://huggingface.co/datasets/BabyLM-community/BabyLM-2026-Strict/resolve/main/open_subtitles.train.txt",
    "hi": "https://huggingface.co/datasets/pulipakav-1/translated-babylm-hindi/resolve/main/train/open_subtitles.train.hi.txt",
    "te": "https://huggingface.co/datasets/pulipakav-1/translated-babylm-telugu/resolve/main/train/open_subtitles.train.te.txt",
}

def normalize(line):
    line = line.strip().lower()
    return re.sub(r"\s+", " ", line)

babylm_sets = {}
for lang, url in BABYLM_OS_URLS.items():
    print(f"Downloading BabyLM open_subtitles training data ({lang})...")
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    babylm_sets[lang] = {normalize(l) for l in resp.text.splitlines() if normalize(l)}
    print(f"  [{lang}] {len(babylm_sets[lang]):,} unique normalized lines")

In [ ]:
# Cell 4: filter OPUS-100 en-hi (English + Hindi sides) and en-te (Telugu side) down to
# the genuinely non-overlapping subset for each language. English uses en-hi's English
# side specifically, matching OS_OPUS_CONFIG in eval_gptbert_all.py.
from datasets import load_dataset

clean_texts = {"en": [], "hi": [], "te": []}
totals = {"en": 0, "hi": 0, "te": 0}

print("Streaming OPUS-100 en-hi (English + Hindi)...")
ds = load_dataset("Helsinki-NLP/opus-100", "en-hi", split="train", streaming=True)
for row in ds:
    en_text = row["translation"]["en"].strip()
    hi_text = row["translation"]["hi"].strip()
    if en_text:
        totals["en"] += 1
        if normalize(en_text) not in babylm_sets["en"]:
            clean_texts["en"].append(en_text)
    if hi_text:
        totals["hi"] += 1
        if normalize(hi_text) not in babylm_sets["hi"]:
            clean_texts["hi"].append(hi_text)

print("Streaming OPUS-100 en-te (Telugu)...")
ds = load_dataset("Helsinki-NLP/opus-100", "en-te", split="train", streaming=True)
for row in ds:
    te_text = row["translation"]["te"].strip()
    if te_text:
        totals["te"] += 1
        if normalize(te_text) not in babylm_sets["te"]:
            clean_texts["te"].append(te_text)

for lang in ["en", "hi", "te"]:
    print(f"[{lang}] Full: {totals[lang]:,} | Clean: {len(clean_texts[lang]):,} "
          f"({100*len(clean_texts[lang])/max(totals[lang],1):.1f}% retained)")

In [ ]:
# Cell 5: shared causal perplexity function for standard HF causal LMs -- same method as
# multilingual eval/eng_hin.py and eval_llama_missing.py: micro-averaged, max_seq_len=128,
# batch_size=32, matching the methodology used for every other perplexity number in this
# project.
import math
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_SEQ_LEN = 128
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def compute_causal_perplexity_hf(model, tokenizer, texts, max_seq_len=MAX_SEQ_LEN, batch_size=BATCH_SIZE):
    total_nll, total_tokens = 0.0, 0
    for i in tqdm(range(0, len(texts), batch_size), desc="  perplexity"):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_len)
        input_ids = enc["input_ids"].to(DEVICE)
        attn_mask = enc["attention_mask"].to(DEVICE)
        with torch.no_grad():
            logits = model(input_ids, attention_mask=attn_mask).logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        shift_mask = attn_mask[:, 1:].contiguous().float()
        log_probs = F.log_softmax(shift_logits, dim=-1)
        token_ll = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)
        total_nll += -(token_ll * shift_mask).sum().item()
        total_tokens += shift_mask.sum().item()
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float("inf")

In [ ]:
# Cell 6: clean OS-data perplexity for standard HF causal LMs -- GPT-Wee (mono-en/hi/te,
# eng-hin, eng-tel), Llama 3.2 1B, Sarvam-2B, across every language each model was
# evaluated on in evaluation.md
HF_MODEL_SPECS = [
    ("GPT-Wee mono-en", "pulipakav-1/gpt2-english-babylm2026", ["en"]),
    ("GPT-Wee mono-hi", "pulipakav-1/gpt2-hindi-babylm2026", ["hi"]),
    ("GPT-Wee mono-te", "pulipakav-1/gpt2-telugu-babylm2026", ["te"]),
    ("GPT-Wee eng-hin", "pulipakav-1/gpt2-hin-eng_babylm2026", ["en", "hi"]),
    ("GPT-Wee eng-tel", "pulipakav-1/gpt2-tel-eng_babylm2026", ["en", "te"]),
    ("Llama 3.2 1B", "meta-llama/Llama-3.2-1B", ["en", "hi", "te"]),
    ("Sarvam-2B", "sarvamai/sarvam-2b-v0.5", ["en", "hi", "te"]),
]

results = {}
for label, repo, langs in HF_MODEL_SPECS:
    print(f"\n=== {label} ({repo}) ===")
    tokenizer = AutoTokenizer.from_pretrained(repo)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(repo).eval().to(DEVICE)
    for lang in langs:
        ppl = compute_causal_perplexity_hf(model, tokenizer, clean_texts[lang])
        print(f"  [{lang}] clean OS-data perplexity: {ppl:.4f}")
        results[f"{label} ({lang})"] = ppl
    del model
    torch.cuda.empty_cache()

In [ ]:
# Cell 7: clean OS-data perplexity for GPT-BERT (mono-en/hi/te, eng-hin, eng-tel) --
# reuses eval_gptbert_all.py's own model-loading and causal-perplexity code directly, so
# the scoring math is guaranteed identical to every other GPT-BERT number already recorded
%cd /content/BabyLM
import sys
sys.path.insert(0, "/content/BabyLM")
from eval_gptbert_all import load_model_and_tokenizer, compute_causal_perplexity, MODEL_REGISTRY

GPTBERT_MODEL_SPECS = [
    ("GPT-BERT mono-en", "mono_en", ["en"]),
    ("GPT-BERT mono-hi", "mono_hi", ["hi"]),
    ("GPT-BERT mono-te", "mono_te", ["te"]),
    ("GPT-BERT eng-hin", "en_hi_seed1", ["en", "hi"]),
    ("GPT-BERT eng-tel", "en_tel_seed2", ["en", "te"]),
]

for label, model_key, langs in GPTBERT_MODEL_SPECS:
    print(f"\n=== {label} ({model_key}) ===")
    spec = MODEL_REGISTRY[model_key]
    model, tokenizer, cls_id, mask_id, pad_id = load_model_and_tokenizer(spec)
    for lang in langs:
        ppl = compute_causal_perplexity(model, tokenizer, clean_texts[lang], cls_id, pad_id)
        print(f"  [{lang}] clean OS-data perplexity: {ppl:.4f}")
        results[f"{label} ({lang})"] = ppl
    del model
    torch.cuda.empty_cache()

In [ ]:
# Cell 8: summary
print("Clean OS-data sizes:")
for lang in ["en", "hi", "te"]:
    print(f"  [{lang}] {len(clean_texts[lang]):,} lines")
print()
for label, ppl in results.items():
    print(f"  {label:28s}: {ppl:.4f}")